<a href="https://colab.research.google.com/github/supunabeywickrama/my-colab-work/blob/main/youtube_transcript_api.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip uninstall -y youtube-transcript-api
!pip install youtube-transcript-api==0.6.1

In [2]:
!apt-get -qq update
!apt-get -qq install -y nodejs npm
!node -v

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
Selecting previously unselected package gyp.
(Reading database ... 121852 files and directories currently installed.)
Preparing to unpack .../000-gyp_0.1+20210831gitd6c5dd5-5_all.deb ...
Unpacking gyp (0.1+20210831gitd6c5dd5-5) ...
Selecting previously unselected package javascript-common.
Preparing to unpack .../001-javascript-common_11+nmu1_all.deb ...
Unpacking javascript-common (11+nmu1) ...
Selecting previously unselected package libjs-events.
Preparing to unpack .../002-libjs-events_3.3.0+~3.0.0-2_all.deb ...
Unpacking libjs-events (3.3.0+~3.0.0-2) ...
Selecting previously unselected package libjs-highlight.js.
Preparing to unpack .../003-libjs-highlight.js_9.18.5+dfsg1-1_all.deb ...
Unpacking libjs-highlight.js (9.18.5+dfsg1-1) ...
Selecting previously 

In [3]:
!pip -q install -U yt-dlp

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.0/182.0 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 77.3 MB/s eta 0:00:00


In [4]:
!ls -la

total 16
drwxr-xr-x 1 root root 4096 Feb  6 14:31 .
drwxr-xr-x 1 root root 4096 Feb 21 08:53 ..
drwxr-xr-x 4 root root 4096 Feb  6 14:31 .config
drwxr-xr-x 1 root root 4096 Feb  6 14:31 sample_data


In [8]:
import re, os, glob, subprocess

def extract_video_id(url: str) -> str:
    m = re.search(r"(?:v=)([0-9A-Za-z_-]{11})", url)
    if not m:
        m = re.search(r"(?:youtu\.be/|shorts/|embed/)([0-9A-Za-z_-]{11})", url)
    if m:
        return m.group(1)
    raise ValueError("Could not extract video id")

def vtt_to_text(vtt: str) -> str:
    lines = []
    for line in vtt.splitlines():
        s = line.strip()
        if not s:
            continue
        if s.startswith("WEBVTT"):
            continue
        if "-->" in s:
            continue
        if re.fullmatch(r"\d+", s):
            continue

        s = re.sub(r"<[^>]+>", "", s)
        s = s.replace("&amp;", "&").replace("&lt;", "<").replace("&gt;", ">")
        s = re.sub(r"\s+", " ", s).strip()
        if s:
            lines.append(s)

    cleaned = []
    prev_norm = ""
    for s in lines:
        s_norm = re.sub(r"[^a-z0-9]+", "", s.lower())
        if s_norm == prev_norm:
            continue
        cleaned.append(s)
        prev_norm = s_norm

    text = "\n".join(cleaned)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    return text

def get_transcript_ytdlp(url: str, lang="en"):
    vid = extract_video_id(url)

    for f in glob.glob(f"subs_{vid}*"):
        try:
            os.remove(f)
        except:
            pass

    cmd = [
        "yt-dlp",
        "--skip-download",
        "--cookies", "cookies.txt",
        "--write-subs",
        "--write-auto-subs",
        "--sub-format", "vtt",
        "--sub-langs", f"{lang}.*",
        "--sleep-requests", "2",
        "--sleep-interval", "2",
        "--max-sleep-interval", "5",
        "-o", f"subs_{vid}.%(ext)s",
        url
    ]

    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(r.stderr.strip() or r.stdout.strip() or "yt-dlp failed")

    vtts = sorted(glob.glob(f"subs_{vid}*.vtt"))
    if not vtts:
        raise RuntimeError("No subtitles found (captions disabled/unavailable).")

    with open(vtts[0], "r", encoding="utf-8", errors="ignore") as f:
        return vtt_to_text(f.read())

def save_transcript(text: str, filename="transcript.txt"):
    with open(filename, "w", encoding="utf-8") as f:
        f.write(text)
    return filename

In [11]:
url = "https://www.youtube.com/watch?v=XtS8mCcffrs&list=PLNWNEEf8BvG64FVZT4IdieI1PuYnHkUrt&index=12"

text = get_transcript_ytdlp(url, lang="en")
print(text[:2000000])
print("\n---\nTotal characters:", len(text))

save_transcript(text, "transcript.txt")
print("Saved: transcript.txt")

Kind: captions
Language: en
Automatic Addison. In this tutorial,
we'll create a ROSS 2 service using C++.
A ROSS 2 service is a way for different
parts of a robot system to communicate
with each other by sending a request and
receiving a response just like making a
phone call and waiting for an answer.
This form of communication is different
from the publish subscribe communication
method where one part of the system
continuously broadcasts information and
any other parts that are interested can
listen in like a radio station
broadcasting to many listeners. With the
service, the communication is direct and
specific where the caller sends a
request to a particular service provider
and waits for a response before
proceeding. In contrast, publish
subscribe allows for more flexible and
asynchronous communication between
multiple parts of the system without
waiting for specific responses. Services
are useful when you need a quick but
specific tasks done and you need to wait
for it to finish